# Module 2 — Analyse et préparation des données

Ce notebook part du fichier propre produit par le module 1. Il ne relit pas les sources brutes et ne refait pas leur nettoyage. Exécuter d'abord `python -m MODULE01.src.main` depuis la racine du dépôt.

In [ ]:
from pathlib import Path
import sys

racine = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "MODULE01").is_dir() and (path / "MODULE02").is_dir()),
    None,
)
if racine is None:
    raise FileNotFoundError("Ouvre ce notebook depuis un dossier du dépôt ImmoAnalyse.")
sys.path.insert(0, str(racine))

import numpy as np
import pandas as pd
from MODULE02.src import config as C, io_sources, stats_eda, viz_eda
from MODULE02.src.pipeline import construire_pipeline, preparer_features
from MODULE02.src.run_pipeline import separer_ensembles

df = io_sources.lire_donnees_module01()
print(f"{len(df)} lignes, {df.shape[1]} colonnes")
display(df.head())

## 1. Explorer avant de transformer

Regarder les distributions et les relations permet de formuler des hypothèses. Une corrélation globale peut masquer des comportements différents par catégorie (paradoxe de Simpson). Mann-Kendall aide à choisir entre une séparation aléatoire et chronologique ; il ne remplace pas l'analyse métier.

In [ ]:
display(stats_eda.comparer_prix_log(df))
display(stats_eda.tester_simpson(df))
tendance = stats_eda.tester_mann_kendall(df)
display(tendance)

In [ ]:
viz_eda.generer_toutes(df, C.FIG_DIR)
print(f"Graphiques enregistrés dans : {C.FIG_DIR}")

## 2. Construire les features sans fuite

`prix_log` est la cible. Le prix, le prix au m² et les informations susceptibles d'être connues après l'annonce ne doivent pas entrer dans `X`. On sépare les données avant d'ajuster l'imputation, le one-hot encoding et la standardisation : ces paramètres sont appris sur train seulement.

In [ ]:
X = preparer_features(df)
y = pd.Series(np.log(df["prix"]), index=df.index, name="prix_log")
strategie = "temporelle" if tendance["p_value"] is not None and tendance["p_value"] < 0.05 else "aleatoire"
ensembles = separer_ensembles(X, y, df["date"], strategie)

preparation = construire_pipeline()
preparation.fit(ensembles["train"][0])
print({nom: len(partie[0]) for nom, partie in ensembles.items()})
print("Nombre de features après encodage :", len(preparation.get_feature_names_out()))

## 3. À retenir

Le module 2 livre un jeu de données exploré et un préprocesseur réutilisable ; il n'entraîne pas encore de modèle de prédiction. Le script `python -m MODULE02.src.run_pipeline` automatise cette chaîne et enregistre les ensembles transformés, le pipeline et les rapports.